# **Prediccion CPU con pesos guardados**

Primero, se prepara el entorno y se recuperan los pesos respaldados en Drive.
Luego, se utilizan imagenes de validacion para poder revisar las predicciones y
el tiempo de respuesta. Se ejecuta solamente forward, sin entrenar ni modificar
los pesos. La particion de prueba permanece reservada.

Este notebook se ejecuta en CPU. Las mediciones son una comprobacion inicial;
los experimentos repetidos de comparacion CPU/GPU se realizan por separado.


## **Preparar el repositorio**

Se ejecuta al iniciar el notebook. Si la carpeta existe, se actualizan las
referencias; si no existe, se clona. Se muestra el commit utilizado. Si cambian
modulos Python ya importados, se reinicia la sesion antes de continuar.


In [ ]:
from pathlib import Path
import subprocess

url_repositorio = "https://github.com/Bamo0507/Proyecto_3_Cuda_CNN.git"
revision_repositorio = "origin/main"
directorio_proyecto = Path("/content/Proyecto_3_Cuda_CNN")

if directorio_proyecto.exists():
    subprocess.run(["git", "fetch", "origin"], cwd=directorio_proyecto, check=True)
else:
    subprocess.run(["git", "clone", url_repositorio, str(directorio_proyecto)], check=True)

subprocess.run(
    ["git", "checkout", "--detach", revision_repositorio],
    cwd=directorio_proyecto,
    check=True,
)
commit_ejecutado = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=directorio_proyecto, text=True
).strip()
print("Commit ejecutado:", commit_ejecutado)


## **Instalar dependencias CPU**

Se instalan las dependencias CPU del repositorio y Matplotlib para mostrar las
imagenes. Si Colab solicita reiniciar la sesion, se reinicia y se vuelve a preparar
el repositorio antes de continuar con Drive.


In [ ]:
import subprocess
import sys

resultado_instalacion = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r",
     str(directorio_proyecto / "requirements.txt"), "matplotlib"],
    capture_output=True,
    text=True,
)
print(resultado_instalacion.stdout, end="")
if resultado_instalacion.stderr:
    print(resultado_instalacion.stderr, end="")
resultado_instalacion.check_returncode()


## **Montar Drive y seleccionar la corrida**

Se autoriza la cuenta de Drive y se selecciona el nombre de la corrida cuyos
pesos se quieren utilizar. La carpeta `proyecto_senias` debe estar en Mi unidad
o tener un acceso directo alli con ese nombre. Para comenzar se utiliza `cpu_01`.


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

directorio_drive = Path("/content/drive/MyDrive/proyecto_senias")
nombre_ejecucion = "cpu_01"


## **Copiar pesos y cargar validacion**

Se copian los pesos, el registro y el NPZ al disco temporal de Colab. Se comprueba
el orden de clases del modelo y del dataset, y se cargan solamente las imagenes
y etiquetas de validacion. Los pesos se recuperan tal como fueron guardados.


In [ ]:
import json
import shutil
import sys
import numpy as np

sys.path.insert(0, str(directorio_proyecto / "src"))

import config
from compartido.persistencia import cargar_parametros
from cpu.prediccion import predecir

directorio_modelo_drive = directorio_drive / "resultados" / nombre_ejecucion
ruta_modelo_local = config.DIRECTORIO_MODELOS / f"{nombre_ejecucion}.npz"
ruta_registro_local = config.DIRECTORIO_RESULTADOS / f"{nombre_ejecucion}.json"

for ruta_origen, ruta_destino in (
    (directorio_modelo_drive / f"{nombre_ejecucion}.npz", ruta_modelo_local),
    (directorio_modelo_drive / f"{nombre_ejecucion}.json", ruta_registro_local),
    (directorio_drive / "02_tensores.npz", config.RUTA_TENSORES),
):
    ruta_destino.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(ruta_origen, ruta_destino)

registro_modelo = json.loads(ruta_registro_local.read_text())
if registro_modelo["configuracion"]["clases"] != list(config.CLASES):
    raise ValueError("El orden de clases del modelo no coincide con config.py.")
if registro_modelo["configuracion"]["forma_entrada"] != list(config.FORMA_ENTRADA):
    raise ValueError("La forma de entrada del modelo no coincide con config.py.")

parametros = cargar_parametros(ruta_modelo_local)
with np.load(config.RUTA_TENSORES, allow_pickle=False) as archivo_tensores:
    if not np.array_equal(archivo_tensores["clases"], np.asarray(config.CLASES)):
        raise ValueError("El orden de clases del dataset no coincide con el modelo.")
    imagenes_validacion = archivo_tensores["x_val"]
    etiquetas_validacion = archivo_tensores["y_val"]

if etiquetas_validacion.shape != (len(imagenes_validacion),):
    raise ValueError("Debe existir una etiqueta por imagen de validacion.")
print("Modelo cargado:", nombre_ejecucion)
print("Imagenes de validacion:", len(imagenes_validacion))


## **Realizar la primera prediccion y calentar Numba**

Se selecciona una imagen de validacion. La primera llamada puede incluir
compilacion de Numba, por lo que su tiempo se registra aparte. Se muestran la
etiqueta real, la prediccion y la probabilidad asignada a la clase seleccionada.


In [ ]:
import time

indice_imagen = 0
imagen_individual = imagenes_validacion[indice_imagen:indice_imagen + 1]

inicio_calentamiento = time.perf_counter()
prediccion_individual = predecir(imagen_individual, parametros)[0]
tiempo_primera_prediccion = time.perf_counter() - inicio_calentamiento

print("Etiqueta real:", config.CLASES[int(etiquetas_validacion[indice_imagen])])
print("Prediccion:", prediccion_individual["etiqueta"])
print(f"Probabilidad: {prediccion_individual['probabilidad']:.2%}")
print(f"Primera prediccion y posible compilacion: {tiempo_primera_prediccion:.4f} s")


## **Revisar ejemplos de validacion**

Se seleccionan cinco ejemplos con semilla fija y se muestran sus etiquetas reales
y predichas. El color permite distinguir aciertos y errores; estos ejemplos no
sustituyen la accuracy calculada sobre toda la validacion.


In [ ]:
import matplotlib.pyplot as graficos

cantidad_ejemplos = min(5, len(imagenes_validacion))
generador_aleatorio = np.random.default_rng(config.SEMILLA_ALEATORIA)
indices_ejemplos = generador_aleatorio.choice(
    len(imagenes_validacion), size=cantidad_ejemplos, replace=False
)
predicciones_ejemplos = predecir(imagenes_validacion[indices_ejemplos], parametros)

figura, ejes = graficos.subplots(
    1, cantidad_ejemplos, figsize=(3 * cantidad_ejemplos, 3), squeeze=False
)
for indice_ejemplo, indice_validacion in enumerate(indices_ejemplos):
    prediccion = predicciones_ejemplos[indice_ejemplo]
    etiqueta_real = config.CLASES[int(etiquetas_validacion[indice_validacion])]
    eje_imagen = ejes[0, indice_ejemplo]
    eje_imagen.imshow(imagenes_validacion[indice_validacion, :, :, 0], cmap="gray", vmin=0, vmax=1)
    eje_imagen.set_title(
        f"Real: {etiqueta_real}\nPred: {prediccion['etiqueta']} ({prediccion['probabilidad']:.1%})",
        color="green" if prediccion["etiqueta"] == etiqueta_real else "red",
    )
    eje_imagen.axis("off")
graficos.tight_layout()
graficos.show()


## **Medir la prediccion despues del calentamiento**

Se repite la prediccion de la misma imagen 30 veces y se registra cada tiempo.
La medicion incluye forward y seleccion de etiqueta, con pesos e imagen en RAM;
excluye Drive, carga de pesos y compilacion inicial. Es una comprobacion de latencia
caliente sobre una imagen, no el experimento final con entradas variadas.


In [ ]:
cantidad_repeticiones = 30
tiempos_prediccion = []

for numero_repeticion in range(cantidad_repeticiones):
    inicio_prediccion = time.perf_counter()
    prediccion_medida = predecir(imagen_individual, parametros)[0]
    tiempos_prediccion.append(time.perf_counter() - inicio_prediccion)

tiempos_milisegundos = np.asarray(tiempos_prediccion) * 1000
print("Repeticiones:", cantidad_repeticiones)
print(f"Latencia promedio: {tiempos_milisegundos.mean():.4f} ms")
print(f"Desviacion estandar: {tiempos_milisegundos.std(ddof=1):.4f} ms")
print("Tiempos individuales (ms):", tiempos_milisegundos.tolist())
